# Platt Scaling Check

Quick sanity check: does Platt scaling improve log loss for `xgb_with_elo`?

**Protocol:**
- Generate walk-forward OOF raw predictions (2020–2024), same as forecasting_results
- Fit Platt scaling on OOF logits: `logit(p_cal) = a + b * logit(p_raw)`
- Report OOF log loss: raw vs calibrated (in-sample for calibrator)
- Apply the OOF-fitted calibrator to 2025 holdout predictions (out-of-sample for calibrator)
- Compare raw vs calibrated on holdout

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("../..").resolve()
GOLD_DIR     = PROJECT_ROOT / "data" / "gold"

# --- Frozen hyperparameters ---
XGB_PARAMS = {
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "max_depth":        6,
    "min_child_weight": 3,
    "subsample":        0.8,
    "colsample_bytree": 0.6,
    "reg_lambda":       1.0,
    "reg_alpha":        0.0,
    "gamma":            0.1,
    "learning_rate":    0.02,
    "seed":             42,
    "nthread":         -1,
}
NUM_BOOST_ROUND       = 3000
EARLY_STOPPING_ROUNDS = 150
N_PLAYERS             = 7
CLIP_EPS              = 1e-6
OOF_YEARS             = list(range(2020, 2025))
TRAIN_START           = 2015

In [2]:
# --- Feature columns ---
PLAYER_MODEL_FEATURES = [
    "m_ewma_pre", "q_pre", "days_since_first_report_pre",
    "days_since_last_dnp_pre", "consec_dnps_pre", "played_last_game_pre",
    "minutes_last_game_pre", "days_since_last_played_pre",
    "injury_present_flag_pre",
]
RECENT_FORM_FEATURES = [
    "net_rtg_ewma_pre", "efg_ewma_pre", "tov_pct_ewma_pre",
    "orb_pct_ewma_pre", "ftr_ewma_pre",
]
STYLE_FEATURES = [
    "off_3pa_rate_pre", "def_3pa_allowed_pre", "off_2pa_rate_pre",
    "def_2pa_allowed_pre", "off_tov_pct_pre", "def_forced_tov_pre",
]
SCHEDULE_FEATURES = [
    "days_rest_pre", "is_b2b_pre", "games_last_4_days_pre",
    "games_last_7_days_pre", "travel_miles_pre", "timezone_shift_hours_pre",
]

def build_feature_cols(n_players):
    cols = []
    for side in ("home", "away"):
        for slot in range(1, n_players + 1):
            for feat in PLAYER_MODEL_FEATURES:
                cols.append(f"{side}_p{slot}_{feat}")
    for feat in RECENT_FORM_FEATURES:
        cols.append(f"home_{feat}")
        cols.append(f"away_{feat}")
    for feat in STYLE_FEATURES:
        cols.append(f"home_{feat}")
        cols.append(f"away_{feat}")
    for feat in SCHEDULE_FEATURES:
        cols.append(f"home_{feat}")
        cols.append(f"away_{feat}")
    return cols

FEATURE_COLS = build_feature_cols(N_PLAYERS)
print(f"Features: {len(FEATURE_COLS)}")

Features: 160


In [3]:
# --- Helpers ---
def clip(p):
    return np.clip(p, CLIP_EPS, 1 - CLIP_EPS)

def logit(p):
    p = clip(p)
    return np.log(p / (1 - p))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def load_gold(season):
    p = GOLD_DIR / f"game_xgboost_input_{season}_REGPST.csv"
    df = pd.read_csv(p)
    df["season"] = df["season"].astype(int)
    cold = (df["home_p1_m_ewma_pre"] == 0) | (df["away_p1_m_ewma_pre"] == 0)
    n_drop = cold.sum()
    if n_drop:
        df = df[~cold].reset_index(drop=True)
        print(f"  [load {season}] dropped {n_drop} cold-start rows")
    return df

def load_range(start, end):
    dfs = []
    for y in range(start, end + 1):
        try:
            dfs.append(load_gold(y))
        except FileNotFoundError as e:
            print(f"  WARNING: {e}")
    return pd.concat(dfs, ignore_index=True).dropna(subset=["home_win", "base_margin"])

def fit_predict_xgb(X_train, y_train, X_es_val, y_es_val,
                     X_test, bm_train, bm_es_val, bm_test, feature_names=None):
    def make_dm(X, y, bm):
        kw = dict(data=X.astype(float), label=y.astype(float),
                  feature_names=feature_names, missing=np.nan)
        kw["base_margin"] = bm.astype(float)
        return xgb.DMatrix(**kw)

    dm_es_train = make_dm(X_train, y_train, bm_train)
    dm_es_val   = make_dm(X_es_val, y_es_val, bm_es_val)

    es_model = xgb.train(
        XGB_PARAMS, dm_es_train,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dm_es_val, "val")],
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=False,
    )
    best_round = es_model.best_iteration + 1

    X_full  = np.vstack([X_train, X_es_val])
    y_full  = np.concatenate([y_train, y_es_val])
    bm_full = np.concatenate([bm_train, bm_es_val])
    dm_full = make_dm(X_full, y_full, bm_full)
    dm_test = make_dm(X_test, np.zeros(len(X_test)), bm_test)

    model = xgb.train(XGB_PARAMS, dm_full, num_boost_round=best_round, verbose_eval=False)
    preds = model.predict(dm_test)
    return clip(preds), best_round

## Step 1 — Generate OOF raw predictions (walk-forward, 2020–2024)

In [4]:
oof_records = []

for test_year in OOF_YEARS:
    print(f"Fold: test_year={test_year}")
    es_val_year = test_year - 1

    es_train_df = load_range(TRAIN_START, es_val_year - 1)
    es_val_df   = load_gold(es_val_year).dropna(subset=["home_win", "base_margin"])
    test_df     = load_gold(test_year).dropna(subset=["home_win", "base_margin"])

    avail = [c for c in FEATURE_COLS if c in es_train_df.columns]

    preds, br = fit_predict_xgb(
        es_train_df[avail].values, es_train_df["home_win"].values.astype(float),
        es_val_df[avail].values,   es_val_df["home_win"].values.astype(float),
        test_df[avail].values,
        es_train_df["base_margin"].values.astype(float),
        es_val_df["base_margin"].values.astype(float),
        test_df["base_margin"].values.astype(float),
        feature_names=avail,
    )
    print(f"  best_round={br}")

    for i in range(len(test_df)):
        oof_records.append({
            "game_id":  test_df.iloc[i]["game_id"],
            "season":   int(test_df.iloc[i]["season"]),
            "home_win": int(test_df.iloc[i]["home_win"]),
            "p_raw":    float(preds[i]),
            "p_elo":    float(test_df.iloc[i]["p_elo"]),
        })

oof = pd.DataFrame(oof_records)
print(f"\nOOF rows: {len(oof)}")

Fold: test_year=2020
  [load 2015] dropped 7 cold-start rows
  best_round=10
Fold: test_year=2021
  [load 2015] dropped 7 cold-start rows
  best_round=63
Fold: test_year=2022
  [load 2015] dropped 7 cold-start rows
  best_round=26
Fold: test_year=2023
  [load 2015] dropped 7 cold-start rows
  best_round=25
Fold: test_year=2024
  [load 2015] dropped 7 cold-start rows
  best_round=87

OOF rows: 1117


## Step 2 — Fit Platt scaling on OOF, report OOF raw vs calibrated

In [5]:
# Fit Platt scaling: logit(p_cal) = a + b * logit(p_raw)
# Using LogisticRegression on the logit of raw predictions
logit_raw_oof = logit(oof["p_raw"].values).reshape(-1, 1)
y_oof = oof["home_win"].values.astype(float)

platt = LogisticRegression(penalty=None, solver="lbfgs", max_iter=5000, fit_intercept=True)
platt.fit(logit_raw_oof, y_oof)

a = platt.intercept_[0]
b = platt.coef_[0, 0]
print(f"Platt parameters:  a = {a:.6f},  b = {b:.6f}")
print(f"  (perfect calibration would be a=0, b=1)")

# Apply to OOF
oof["p_cal"] = sigmoid(a + b * logit(oof["p_raw"].values))

# Metrics
p_elo_oof = clip(oof["p_elo"].values)
ll_elo = log_loss(y_oof, p_elo_oof)
ll_raw = log_loss(y_oof, clip(oof["p_raw"].values))
ll_cal = log_loss(y_oof, clip(oof["p_cal"].values))

br_elo = brier_score_loss(y_oof, p_elo_oof)
br_raw = brier_score_loss(y_oof, clip(oof["p_raw"].values))
br_cal = brier_score_loss(y_oof, clip(oof["p_cal"].values))

print(f"\n{'OOF 2020-2024':>20s}  {'Log Loss':>10s}  {'Brier':>10s}")
print(f"{'Elo only':>20s}  {ll_elo:10.5f}  {br_elo:10.5f}")
print(f"{'XGB raw':>20s}  {ll_raw:10.5f}  {br_raw:10.5f}")
print(f"{'XGB + Platt':>20s}  {ll_cal:10.5f}  {br_cal:10.5f}")
print(f"\n  Platt LL improvement on OOF: {ll_raw - ll_cal:+.6f}")
print(f"  (Note: this is in-sample for the calibrator)")

Platt parameters:  a = -0.043139,  b = 0.975050
  (perfect calibration would be a=0, b=1)

       OOF 2020-2024    Log Loss       Brier
            Elo only     0.60216     0.20724
             XGB raw     0.59939     0.20550
         XGB + Platt     0.59912     0.20545

  Platt LL improvement on OOF: +0.000275
  (Note: this is in-sample for the calibrator)


## Step 3 — 2025 Holdout: apply OOF-fitted Platt to holdout predictions

In [6]:
# Train holdout model: ES on 2015-2023, val on 2024, retrain on 2015-2024, predict 2025
es_train_h_df = load_range(TRAIN_START, 2023)
es_val_h_df   = load_gold(2024).dropna(subset=["home_win", "base_margin"])
holdout_df    = load_gold(2025).dropna(subset=["home_win", "base_margin"])

avail = [c for c in FEATURE_COLS if c in es_train_h_df.columns]

preds_h, br_h = fit_predict_xgb(
    es_train_h_df[avail].values, es_train_h_df["home_win"].values.astype(float),
    es_val_h_df[avail].values,   es_val_h_df["home_win"].values.astype(float),
    holdout_df[avail].values,
    es_train_h_df["base_margin"].values.astype(float),
    es_val_h_df["base_margin"].values.astype(float),
    holdout_df["base_margin"].values.astype(float),
    feature_names=avail,
)
print(f"Holdout best_round={br_h}")

y_hold    = holdout_df["home_win"].values.astype(float)
p_elo_h   = clip(holdout_df["p_elo"].values.astype(float))
p_raw_h   = clip(preds_h)
p_cal_h   = sigmoid(a + b * logit(p_raw_h))

ll_elo_h = log_loss(y_hold, p_elo_h)
ll_raw_h = log_loss(y_hold, p_raw_h)
ll_cal_h = log_loss(y_hold, clip(p_cal_h))

br_elo_h = brier_score_loss(y_hold, p_elo_h)
br_raw_h = brier_score_loss(y_hold, p_raw_h)
br_cal_h = brier_score_loss(y_hold, clip(p_cal_h))

print(f"\n{'2025 Holdout':>20s}  {'Log Loss':>10s}  {'Brier':>10s}")
print(f"{'Elo only':>20s}  {ll_elo_h:10.5f}  {br_elo_h:10.5f}")
print(f"{'XGB raw':>20s}  {ll_raw_h:10.5f}  {br_raw_h:10.5f}")
print(f"{'XGB + Platt':>20s}  {ll_cal_h:10.5f}  {br_cal_h:10.5f}")
print(f"\n  Platt LL improvement on holdout: {ll_raw_h - ll_cal_h:+.6f}")
print(f"  (This IS out-of-sample for the calibrator)")

  [load 2015] dropped 7 cold-start rows
Holdout best_round=88

        2025 Holdout    Log Loss       Brier
            Elo only     0.61510     0.21317
             XGB raw     0.61215     0.21123
         XGB + Platt     0.61298     0.21160

  Platt LL improvement on holdout: -0.000834
  (This IS out-of-sample for the calibrator)


## Step 4 — Summary comparison table

In [7]:
summary = pd.DataFrame([
    {"Period": "OOF 2020-2024", "Model": "Elo only",    "Log Loss": ll_elo,   "Brier": br_elo},
    {"Period": "OOF 2020-2024", "Model": "XGB raw",     "Log Loss": ll_raw,   "Brier": br_raw},
    {"Period": "OOF 2020-2024", "Model": "XGB + Platt", "Log Loss": ll_cal,   "Brier": br_cal},
    {"Period": "2025 Holdout",  "Model": "Elo only",    "Log Loss": ll_elo_h, "Brier": br_elo_h},
    {"Period": "2025 Holdout",  "Model": "XGB raw",     "Log Loss": ll_raw_h, "Brier": br_raw_h},
    {"Period": "2025 Holdout",  "Model": "XGB + Platt", "Log Loss": ll_cal_h, "Brier": br_cal_h},
])
print(summary.to_string(index=False, float_format="%.5f"))

print(f"\nPlatt parameters: a={a:.6f}, b={b:.6f}")
print(f"If b≈1 and a≈0, raw predictions are already well-calibrated.")

       Period       Model  Log Loss   Brier
OOF 2020-2024    Elo only   0.60216 0.20724
OOF 2020-2024     XGB raw   0.59939 0.20550
OOF 2020-2024 XGB + Platt   0.59912 0.20545
 2025 Holdout    Elo only   0.61510 0.21317
 2025 Holdout     XGB raw   0.61215 0.21123
 2025 Holdout XGB + Platt   0.61298 0.21160

Platt parameters: a=-0.043139, b=0.975050
If b≈1 and a≈0, raw predictions are already well-calibrated.
